In [ ]:
import pandas as pd, re, matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image
import numpy as np
import torch, torchvision.transforms as T, matplotlib.pyplot as plt, numpy as np, json
from torchvision.models import resnet18
import re
import matplotlib.colors as mcolors

In [ ]:
r = Path.cwd().parent.parent
MODEL_PATH = r / "models/two_models_v2/class_aug_test_light_vs_heavy/DS_StdAll_best.pth"
LOG_FILE = r / Path("data/test/Харасавэйск_1700 к17_посл_лит_опис_(93,35 м).csv")
DS_DIR   = r / Path("data/test/ДС")
UV_DIR   = r / Path("data/test/УФ")
ENC_PATH   = r / "data\Digital_core_tmv2\label_encoder.json"
df = pd.read_csv(LOG_FILE).sort_values("depth_from")

In [ ]:
## Какие именно куски керна будет кгадывать модель нужно указать в конце второй ячейки.

In [ ]:
PALETTE = {
    "Песчаник": "#3B82F6", "Песчаник карбонатный": "#14B8A6",
    "Аргиллит": "#EF4444", "Аргиллит с прослоями песчаника и алевролита": "#C73E1D",
    "Алевролит": "#F59E0B", "Алевролит с включениями угля": "#D97D00",
    "Переслаивание песчаника, аргиллита и алевролита": "#8B5CF6",
    "Глинисто-карбонатная порода": "#78716C", "Глина аргиллитоподобная": "#A855F7",
    "Уголь": "#292524", "unknown": "#E5E7EB"
}

In [ ]:
def get_depths(fname):
    m = re.search(r'(\d+\.\d+)\s*-\s*(\d+\.\d+)', fname)
    return (float(m.group(1)), float(m.group(2))) if m else (None, None)

def plot_core_with_legend(z0, z1):
    # Ищем файлы с допуском 0.01 (для точности float)
    ds = next((p for p in DS_DIR.glob("*.jpeg") if get_depths(p.name) and abs(get_depths(p.name)[0] - z0) < 0.01), None)
    uv = next((p for p in UV_DIR.glob("*.jpeg") if get_depths(p.name) and abs(get_depths(p.name)[0] - z0) < 0.01), None)
    
    if not ds or not uv:
        print(f"Нет фото для интервала {z0}-{z1} м"); return
    
    img_ds = np.array(Image.open(ds).convert("RGB"))
    img_uv = np.array(Image.open(uv).convert("RGB"))
    
    # Широкий формат
    fig, (ax1, ax2, ax3, ax_leg) = plt.subplots(1, 4, figsize=(20, 9), 
                                                 gridspec_kw={'width_ratios': [1, 1, 1.8, 2.2], 'wspace': 0.2})
    fig.suptitle(f"Интервал {z0:.2f} — {z1:.2f} м", fontsize=14, fontweight='bold', y=0.96)
    
    extent = [0, 1, z1, z0]
    ax1.imshow(img_ds, aspect='auto', extent=extent); ax1.axis('off'); ax1.set_title("ДС", fontsize=12)
    ax2.imshow(img_uv, aspect='auto', extent=extent); ax2.axis('off'); ax2.set_title("УФ", fontsize=12)
    
    ax3.set_xlim(0, 1); ax3.set_ylim(z1, z0)
    subset = df[(df.depth_from < z1) & (df.depth_to > z0)]
    
    for _, row in subset.iterrows():
        y0, y1 = max(row['depth_from'], z0), min(row['depth_to'], z1)
        if y1 - y0 < 0.02: continue
        color = PALETTE.get(row['mineral'].strip(), PALETTE['unknown'])
        ax3.fill_betweenx([y0, y1], 0.08, 0.92, color=color, linewidth=0)
    
    ax3.set_xticks([]); ax3.set_yticks([z0, z1]); ax3.set_title("Разметка", fontsize=12)
    for s in ax3.spines.values(): s.set_visible(False)
    
    ax_leg.axis('off'); ax_leg.set_title("Легенда", fontsize=13, fontweight='bold')
    present_minerals = sorted(set(row['mineral'].strip() for _, row in subset.iterrows()))
    y_pos = 0.93
    for mineral in present_minerals:
        color = PALETTE.get(mineral, PALETTE['unknown'])
        ax_leg.add_patch(plt.Rectangle((0.05, y_pos-0.025), 0.18, 0.045, facecolor=color, edgecolor='#444', lw=0.8))
        ax_leg.text(0.28, y_pos-0.005, mineral, fontsize=11, va='center')
        y_pos -= 0.08
    ax_leg.set_ylim(0, 1); ax_leg.set_xlim(0, 1)
    plt.subplots_adjust(top=0.92, bottom=0.05)
    plt.show()

# ─── ЗАПУСК (ЗДЕСЬ МЕНЯТЬ ГЛУБИНУ) ───────────────────────────────────────
print("Показываю интервалы 1718.7 и 1721.2:")
plot_core_with_legend(1718.7, 1719.7)
plot_core_with_legend(1721.2, 1722.2)



In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:

# ─── 2. ЗАГРУЗКА НАЗВАНИЙ КЛАССОВ ───────────────────────────────────────────

CLASS_NAMES = {}
if ENC_PATH.exists():
    with open(ENC_PATH, 'r', encoding='utf-8') as f:
        raw = json.load(f)
        
        # Твой файл: {"Песчаник": 0, ...} → Нам нужно: {0: "Песчаник", ...}
        first_val = next(iter(raw.values()))
        if isinstance(first_val, int):
            CLASS_NAMES = {v: k for k, v in raw.items()}  # Меняем ключи и значения местами
        else:
            CLASS_NAMES = {int(k): v for k, v in raw.items()}  # Если уже {0: "Песчаник"}
            
    print(f"Загружено {len(CLASS_NAMES)} классов: {list(CLASS_NAMES.values())[:3]}...")
else:
    CLASS_NAMES = {i: f"Класс_{i}" for i in range(9)}
    print("label_encoder.json не найден, использую заглушки")

# ─── 3. ТРАНСФОРМЫ ─────────────────────────────────────────────────────────
class PadToSquare:
    def __call__(self, img):
        w, h = img.size
        if w == h: return img
        m = max(w, h)
        return T.functional.pad(img, ((m-w)//2, (m-h)//2, (m-w+1)//2, (m-h+1)//2), fill=128)

preprocess = T.Compose([
    PadToSquare(), T.Resize((224, 224)), T.ToTensor(),
    T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# ─── 4. ЗАГРУЗКА МОДЕЛИ ────────────────────────────────────────────────────
ckpt = torch.load(MODEL_PATH, map_location=device, weights_only=False)
num_classes = next((v.shape[0] for k,v in ckpt.items() if 'fc.weight' in k), 9)
model = resnet18(weights=None)
model.fc = torch.nn.Linear(model.fc.in_features, num_classes)
clean_ckpt = {k.replace('module.','').replace('model.',''): v for k,v in ckpt.items()}
model.load_state_dict(clean_ckpt, strict=False)
model.to(device).eval()

# ─── 5. ФИКСИРОВАННЫЕ ЦВЕТА (чтобы легенда всегда совпадала) ───────────────
CLASS_COLORS = {
    0: "#3B82F6", 1: "#EF4444", 2: "#F59E0B", 3: "#8B5CF6",
    4: "#78716C", 5: "#292524", 6: "#14B8A6", 7: "#A855F7", 8: "#94A3B8"
}

# ─── 6. ВИЗУАЛИЗАЦИЯ ───────────────────────────────────────────────────────
def show_predictions_with_names(depth_start):
    # Поиск файла
    target = float(depth_start)
    img_path = next((p for p in DS_DIR.glob("*.jpeg") 
                     if re.search(r'(\d+\.\d+)', p.name) and 
                     abs(float(re.search(r'(\d+\.\d+)', p.name).group(1)) - target) < 0.01), None)
    if not img_path: return print(f"⚠️ Не найдено фото для {depth_start}")
    
    img = np.array(Image.open(img_path).convert("RGB"))
    h = img.shape[0]
    chunk_px = h // 20  # 20 кусков по 5 см
    
    # Предсказания
    preds = []
    for i in range(20):
        y1, y2 = i*chunk_px, (i+1)*chunk_px
        crop = Image.fromarray(img[y1:y2])
        with torch.no_grad():
            out = model(preprocess(crop).unsqueeze(0).to(device))
            probs = torch.softmax(out, 1)
            conf, idx = torch.max(probs, 1)
        preds.append((idx.item(), conf.item()))
    
    # 🎨 Рисуем: [Фото] [Предсказания] [Легенда]
    fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(16, 10), 
                                        gridspec_kw={'width_ratios': [2, 1, 1.2], 'wspace': 0.15})
    fig.suptitle(f"{depth_start} — {float(depth_start)+1:.1f} м", fontsize=12, fontweight='bold', y=0.98)
    
    # Фото
    extent = [0, 1, float(depth_start)+1, float(depth_start)]
    ax1.imshow(img, aspect='auto', extent=extent)
    ax1.axis('off')
    ax1.set_title("Фото ДС", fontsize=10)
    
    # Предсказания (цветные блоки с названиями)
    ax2.set_xlim(0, 1); ax2.set_ylim(float(depth_start)+1, float(depth_start))
    for i, (cls_idx, conf) in enumerate(preds):
        d0 = float(depth_start) + i*0.05
        d1 = d0 + 0.05
        color = CLASS_COLORS.get(cls_idx, "#E5E7EB")
        ax2.fill_betweenx([d0, d1], 0.1, 0.9, color=color, linewidth=0)
        # Подпись: первая буква названия + уверенность (если место есть)
        if conf > 0.5 and (d1-d0) > 0.03:
            name = CLASS_NAMES.get(cls_idx, f"C{cls_idx}")
            label = f"{name.split()[0]}\n{conf:.0%}" if len(name) < 15 else f"{name[:8]}..\n{conf:.0%}"
            ax2.text(0.5, (d0+d1)/2, label, ha='center', va='center', 
                    fontsize=7, color='white', fontweight='bold', rotation=90)
    ax2.set_xticks([])
    ax2.set_yticks([float(depth_start), float(depth_start)+1])
    ax2.set_yticklabels([f"{depth_start}", f"{float(depth_start)+1:.1f}"], fontsize=9)
    ax2.set_title("Модель", fontsize=10)
    for s in ax2.spines.values(): s.set_visible(False)
    
    # Легенда с реальными названиями
    ax3.axis('off')
    ax3.set_title("Легенда классов", fontsize=10, fontweight='bold', pad=10)
    for idx in range(num_classes):
        name = CLASS_NAMES.get(idx, f"Класс {idx}")
        color = CLASS_COLORS.get(idx, "#E5E7EB")
        ax3.add_patch(plt.Rectangle((0.05, 0.95 - idx*0.11), 0.25, 0.09, 
                                   facecolor=color, edgecolor='#444', lw=0.8))
        ax3.text(0.35, 0.995 - idx*0.11, name, fontsize=9, va='center', fontweight='500')
    ax3.set_ylim(0, 1); ax3.set_xlim(0, 1)
    
    plt.subplots_adjust(top=0.94, bottom=0.05)
    plt.show()

# ─── 7. ЗАПУСК ──────────────────────────────────────────────────────────────
show_predictions_with_names("1718.7")
show_predictions_with_names("1721.2")